# Module 1: API Configuration and Setup

In [ ]:
import json
import pandas as pd
from datetime import datetime
import os
from tqdm import tqdm
import time

def setup_api_config():
    """
    Configure API settings and parameters for data collection
    Returns a dictionary containing all necessary configuration
    """
    config = {
        "apiEndpoint": "https://api.apify.com/v2/acts/apify~twitter-scraper/runs",
        "apiKey": "",
        "searchParams": {
            "searchTerms": ["black lives matter"],
            "dateRange": {
                "start": "2020-05-01",
                "end": "2021-05-05"
            },
            "tweetsFilter": {
                "minimumRetweets": 5,
                "minimumLikes": 5,
                "excludeReplies": True,
                "excludeRetweets": False
            },
            "languageFilter": "en",
            "searchMode": "Latest",
            "maxTweets": 100000,
            "userFilter": {
                "verifiedOnly": False,
                "minimumFollowers": 0,
                "blueSubscribersOnly": False
            }
        }
    }
    return config

# Module 2: API Request Handler

In [ ]:
def make_api_request(config):
    """
    Make API request to collect Twitter data
    Args:
        config: Dictionary containing API configuration
    Returns:
        API response data
    """
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {config['apiKey']}"
    }

    try:
        response = requests.post(
            config['apiEndpoint'],
            headers=headers,
            json=config['searchParams']
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error making API request: {e}")
        return None


# Module 3: Data Processing


In [ ]:
def process_tweet_data(raw_data):
    """
    Process raw tweet data into structured format
    Args:
        raw_data: Raw API response data
    Returns:
        Processed tweet data in structured format
    """
    processed_tweets = []

    for tweet in raw_data.get('data', []):
        processed_tweet = {
            'type': 'tweet',
            'id': tweet['id'],
            'url': f"https://x.com/{tweet['author']['userName']}/status/{tweet['id']}",
            'text': tweet['text'],
            'retweetCount': tweet.get('retweetCount', 0),
            'replyCount': tweet.get('replyCount', 0),
            'likeCount': tweet.get('likeCount', 0),
            'quoteCount': tweet.get('quoteCount', 0),
            'createdAt': tweet['createdAt'],
            'author': {
                'userName': tweet['author']['userName'],
                'followers': tweet['author'].get('followers', 0),
                'following': tweet['author'].get('following', 0)
            }
        }
        processed_tweets.append(processed_tweet)

    return processed_tweets



# Module 4: Data Saving


In [ ]:
def save_data(processed_data, output_path):
    """
    Save processed data to JSON file
    Args:
        processed_data: Processed tweet data
        output_path: Path to save the data
    """
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(processed_data, f, ensure_ascii=False, indent=2)
        print(f"Data saved successfully to {output_path}")
    except Exception as e:
        print(f"Error saving data: {e}")


# Module 5: Main Execution

In [ ]:
def main():
    """
    Main execution function to run the data collection process
    """
    print("Starting Twitter data collection process...")

    # Setup configuration
    config = setup_api_config()
    print("Configuration setup complete")

    # Make API request
    print("Making API request...")
    raw_data = make_api_request(config)

    if raw_data:
        # Process data
        print("Processing tweet data...")
        processed_data = process_tweet_data(raw_data)

        # Save data
        output_path = f"twitter_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        save_data(processed_data, output_path)

        # Print summary
        print("\nData Collection Summary:")
        print(f"Total tweets collected: {len(processed_data)}")
        print(f"Date range: {config['searchParams']['dateRange']['start']} to {config['searchParams']['dateRange']['end']}")
        print(f"Output file: {output_path}")
    else:
        print("No data collected due to API request failure")